# Мерей: packed dropout and direct head statistics
Validation of exact candidate e4de822c010b58b530d3e95c57305674eb918e18.
The official harness, task and Soup regression tests are unchanged.
The previous solution is measured once before and once after the official
candidate two-pair run in this same T4 session. Those comparisons are diagnostics,
not official score numbers. All subprocesses terminate before another starts.


In [ ]:
import os, sys, subprocess, json, hashlib, statistics
from pathlib import Path
REPO = 'https://github.com/promaladoi/Decentrathon-6.0.git'
REFERENCE = 'd6dca3ac371b6bf5ac15584cd9afbd027aab4a0b'
CANDIDATE = 'e4de822c010b58b530d3e95c57305674eb918e18'
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True)
print(gpu, flush=True)
assert 'T4' in gpu, 'Choose GPU T4 x2'
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'soup-cli[train]==0.77.0', 'transformers==5.16.1', 'peft==0.20.0',
    'trl==0.29.1', 'bitsandbytes==0.50.2', 'accelerate==1.14.0', 'pytest'], check=True)
import importlib.metadata as metadata
versions = {name: metadata.version(name) for name in
    ('torch', 'triton', 'soup-cli', 'transformers', 'peft', 'trl', 'bitsandbytes', 'accelerate')}
print(versions, flush=True)
assert versions['torch'] == '2.11.0+cu128' and versions['triton'] == '3.6.0'


In [ ]:
roots = {}
for name, commit in [('reference', REFERENCE), ('candidate', CANDIDATE)]:
    checkout = Path('/kaggle/working') / ('merey-' + name + '-' + commit[:12])
    assert not checkout.exists(), 'Use a fresh session for this exact run'
    subprocess.run(['git', 'clone', '-q', '--no-checkout', REPO, str(checkout)], check=True)
    subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', commit], check=True)
    assert subprocess.check_output(['git', '-C', str(checkout), 'rev-parse', 'HEAD'], text=True).strip() == commit
    roots[name] = checkout / 'Мерей'
protected = ['task', 'harness', 'tests']
for directory in protected:
    a, b = roots['reference'] / directory, roots['candidate'] / directory
    files = sorted(p.relative_to(a) for p in a.rglob('*') if p.is_file())
    assert files == sorted(p.relative_to(b) for p in b.rglob('*') if p.is_file())
    assert all((a / p).read_bytes() == (b / p).read_bytes() for p in files), directory
os.chdir(roots['candidate'])
sys.path.insert(0, str(roots['candidate'] / 'harness'))
from common import ENV
def run(side, args, **extra):
    print(side, args, flush=True)
    subprocess.run([sys.executable, *args], cwd=roots[side], env=dict(ENV, **extra), check=True)
run('candidate', ['-m', 'pytest', 'solution', '-q', '-p', 'no:cacheprovider'])
run('candidate', ['-m', 'pytest', 'tests', '-q', '-p', 'no:cacheprovider'], SOUP_APPLY_SOLUTION='1')
for side in roots:
    run(side, ['task/prepare_data.py'])


In [ ]:
run('reference', ['harness/run_task.py', '--only', 'solution', '--out', 'runs_before'])
run('candidate', ['harness/run_task.py', '--repeats', '2'])
run('candidate', ['harness/check_rules.py'])
run('reference', ['harness/run_task.py', '--only', 'solution', '--out', 'runs_after'])


In [ ]:
candidate = json.loads((roots['candidate'] / 'runs/result.json').read_text())
checks = json.loads((roots['candidate'] / 'runs/checks.json').read_text())
reference_runs = []
for folder in ['runs_before', 'runs_after']:
    result = json.loads((roots['reference'] / folder / 'result.json').read_text())
    reference_runs.extend(result['runs']['solution'])
old_time = statistics.median(r['wall_s'] for r in reference_runs)
old_memory = max(r['gpu_peak_mib'] for r in reference_runs)
new = candidate['summary']['solution']
comparison = {'reference_commit': REFERENCE, 'candidate_commit': CANDIDATE,
    'versions': versions, 'reference_wall_s': old_time, 'reference_gpu_peak_mib': old_memory,
    'candidate_wall_s': new['wall_s'], 'candidate_gpu_peak_mib': new['gpu_peak_mib'],
    'speedup_vs_previous': old_time / new['wall_s'],
    'memory_ratio_vs_previous': old_memory / new['gpu_peak_mib'],
    'official_summary': candidate['summary'], 'checks': checks}
out = Path('/kaggle/working/merey_optimization_comparison.json')
out.write_text(json.dumps(comparison, indent=2))
print(json.dumps(comparison, indent=2), flush=True)
print('MEREY_VALIDATION_COMPLETE', flush=True)
